In [26]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [27]:
#MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
#MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
#MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)

In [28]:
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


In [29]:
print(model.device)

cpu


In [30]:
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Embedding dimension: 384


In [31]:
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker

REPOSITORY_URL = "https://github.com/Nikita-Filonov/python-api-tests"

In [32]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 18


In [33]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents = processor.process(documents)

print(f"Number of documents before processing: {len(documents)}")

Number of documents before processing: 18


In [34]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

documents_chunking = chunker.chunk(documents)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 42


In [35]:
# Extract only the text
texts = [document.page_content for document in documents_chunking]

In [36]:
import time
# 5. Generate embeddings on CPU
start_time = time.time()

document_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)
print(f"Number of embeddings: {len(document_embeddings)}")
print(f"Embedding generation time: {time.time() - start_time:.2f} seconds")

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.07it/s]

Embedding shape: (42, 384)
Number of embeddings: 42
Embedding generation time: 1.89 seconds


In [37]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Questions and expected source files.
# Adjust the expected paths to match your repository.
TEST_CASES = [
    {
        "question": "¿Dónde se implementan los clientes que realizan las peticiones HTTP?",
        "expected_files": ["clients/"],
    },
    {
        "question": "¿Dónde se definen los fixtures utilizados por los tests?",
        "expected_files": ["conftest.py", "fixtures/"],
    },
    {
        "question": "¿Cómo se configuran los tests del proyecto?",
        "expected_files": ["pytest.ini", "config.py"],
    },
    {
        "question": "Muéstrame el código que valida las respuestas de la API mediante esquemas.",
        "expected_files": ["schema/"],
    },
    {
        "question": "¿Dónde se implementa la lógica para realizar peticiones a la API?",
        "expected_files": ["clients/"],
    },
]

# Normalize embeddings to make cosine similarity straightforward.
chunk_embeddings = np.asarray(document_embeddings)
chunk_embeddings = chunk_embeddings / np.maximum(
    np.linalg.norm(chunk_embeddings, axis=1, keepdims=True),
    1e-12,
)

TOP_K = 5

for test_case in TEST_CASES:
    question = test_case["question"]
    expected_files = test_case["expected_files"]

    # Embed the query using the same model.
    query_embedding = model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Compute cosine similarity against every chunk.
    similarities = chunk_embeddings @ query_embedding

    # Get the indices of the top-k chunks.
    top_indices = np.argsort(similarities)[::-1][:TOP_K]

    print("\n" + "=" * 90)
    print(f"Question: {question}")
    print(f"Expected files: {expected_files or 'Not specified'}")
    print("-" * 90)

    for rank, idx in enumerate(top_indices, start=1):
        doc = documents_chunking[idx]
        source = doc.metadata.get("source", "Unknown")
        file_path = doc.metadata.get("file_path", source)

        print(
            f"\n[{rank}] Similarity: {similarities[idx]:.4f}"
            f"\nFile: {file_path}"
            f"\nContent:\n{doc.page_content[:700]}"
        )


Question: ¿Dónde se implementan los clientes que realizan las peticiones HTTP?
Expected files: ['clients/']
------------------------------------------------------------------------------------------

[1] Similarity: 0.4161
File: README.md
Content:
## Overview  
This project provides an example of API testing for a [REST API](https://en.wikipedia.org/wiki/REST) using Python. It
incorporates powerful libraries and best practices to ensure maintainability, readability, and efficiency. Key
technologies and methodologies used in this project include:  
- [HTTPX](https://www.python-httpx.org/) – A powerful and efficient HTTP client for making API requests.
- [Pydantic](https://docs.pydantic.dev/latest/) – Used for data validation, deserialization, and serialization.
- [Pydantic Settings](https://docs.pydantic.dev/latest/concepts/pydantic_settings/) – Manages test settings in a
structured and maintainable way.
- [Faker](https://faker.readthe

[2] Similarity: 0.3734
File: clients/event_hooks.

In [52]:
query = "Explicame de que trata este repositorio de codigo"

# 1. MiniLM → embedding de la pregunta
query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

# 2. Similarity search
similarities = chunk_embeddings @ query_embedding
top_indices = np.argsort(similarities)[::-1][:5]

# 3. Recuperar contexto
context = "\n\n".join(
    documents_chunking[i].page_content
    for i in top_indices
)

# 4. Context + query → LLM
prompt = f"""
Answer the question using only the following context.

Context:
{context}

Question:
{query}
"""

print(prompt)


Answer the question using only the following context.

Context:
## Overview  
This project provides an example of API testing for a [REST API](https://en.wikipedia.org/wiki/REST) using Python. It
incorporates powerful libraries and best practices to ensure maintainability, readability, and efficiency. Key
technologies and methodologies used in this project include:  
- [HTTPX](https://www.python-httpx.org/) – A powerful and efficient HTTP client for making API requests.
- [Pydantic](https://docs.pydantic.dev/latest/) – Used for data validation, deserialization, and serialization.
- [Pydantic Settings](https://docs.pydantic.dev/latest/concepts/pydantic_settings/) – Manages test settings in a
structured and maintainable way.
- [Faker](https://faker.readthedocs.io/en/master/) – Generates fake data for testing purposes.
- [Pytest](https://docs.pytest.org/en/stable/) – A full-featured and powerful testing framework in Python.
- [JSON Schema validation](https://python-jsonschema.readthedocs

In [43]:
from langchain_ollama import ChatOllama

print("langchain-ollama importado correctamente")

langchain-ollama importado correctamente


In [53]:
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

In [54]:
import time

start = time.time()

response = llm.invoke(prompt)

print(f"Time: {time.time() - start:.2f} seconds")
print(response.content)

Time: 15.32 seconds
Este repositorio es un ejemplo de testing de APIs REST utilizando Python. Su objetivo es demostrar cómo realizar pruebas de API con herramientas y métodos eficientes, incluyendo:

### Tecnologías clave:
- **HTTPX**: Cliente HTTP poderoso y eficiente para realizar solicitudes a APIs.
- **Pydantic**: Validación de datos, deserialización y serialización de datos.
- **Faker**: Generación de datos falsos para pruebas.
- **Pytest**: Framework de prueba completo en Python.
- **Allure**: Herramienta para generar reportes de prueba (usada en este repositorio).
- **SampleAPIs**: Servicios de APIs fakes (como el FakeBank API) para testing.

### Funcionalidades:
- **Pruebas de API**: Realizar pruebas de endpoints de una API REST.
- **Validación de datos**: Usar Pydantic para validar estructuras de datos.
- **Generación de datos falsos**: Usar Faker para crear datos para pruebas.
- **Reporting**: Generar reportes con Allure para monitorear resultados de las pruebas.
- **Organiza

In [ ]:
import time

# Insert API KEY from .env

"""
from langchain_groq import ChatGroq

llm_groq = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0,
)
"""

In [51]:
response = llm_groq.invoke(prompt)

print(response.content)

El esquema que se utiliza para actualizar una operación es el **`UpdateOperationSchema`**.  
En el código se define de la siguiente manera:

```python
class UpdateOperationSchema(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    debit: float | str | None = Field(default_factory=fake.money)
    credit: float | str | None = Field(default_factory=fake.money)
    category: str | None = Field(default_factory=fake.category)
    description: str | None = Field(default_factory=fake.sentence)
    transaction_date: date | None = Field(alias="transactionDate", default_factory=fake.date)
```

**Campos disponibles**

| Campo            | Tipo permitido | Valor por defecto (generado con `fake`) | Comentario |
|------------------|----------------|----------------------------------------|------------|
| `debit`          | `float`, `str`, `None` | `fake.money` | Monto de débito. |
| `credit`         | `float`, `str`, `None` | `fake.money` | Monto de crédito. |
| `category`       | `